# RT-DETRv2 results: query-based detector under the B1 budget

We present the saved measurements of the RT-DETRv2 experiments next to Faster R-CNN B1 and
the stride-five RetinaNet runs. All runs use the same 4,868 training frames, seed 2026,
batch size 2, resize 640/1024, five epochs and therefore 24,340 image exposures, and the
reported model is always the final epoch. AP values use the 0 to 1 scale and the canonical
COCO protocol with the original category-specific crowd records. Missing records show as
**Not run** and partial records as **Partial**. This notebook only reads JSON records; it
trains and evaluates nothing.

In [ ]:
import json
import sys
from pathlib import Path

import plotly.express as px
import polars as pl
from decouple import config
from IPython.display import Markdown, display

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from hm2.faster_rcnn_experiments import aggregate_loss_curve, load_batch_records
from hm2.rt_detr import DIAGNOSTIC_COLUMNS, LOSS_COLUMNS, LOSS_PARTS, LOSS_TERMS
from hm2.rt_detr_experiments import experiment_label, list_experiments

BATCH_ID = "rt_detr_v2_r50_stride5_5epochs_seed2026"
SAVED_RESULTS = ROOT / "runs/rt_detr" / BATCH_ID
batch_dir = Path(config("RT_DETR_BATCH_DIR", default=str(SAVED_RESULTS))).expanduser()
records = (
    load_batch_records(batch_dir) if batch_dir.is_dir() else {"experiments": [], "probe": None}
)
measured = {item["experiment_id"]: item for item in records["experiments"]}
experiments = [
    measured.get(name, {"experiment_id": name, "status": "Not run", "epochs": []})
    for name in list_experiments(ROOT / "config/rt_detr_baseline.toml")
]
analysis_path = batch_dir / "_analysis/analysis.json"
analysis = json.loads(analysis_path.read_text()) if analysis_path.is_file() else None
detectors = analysis["detectors"] if analysis else []
px.defaults.width = 1000
px.defaults.height = 450
pl.Config.set_tbl_cols(14)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(90)
pl.from_dicts(
    [
        {"Experiment": experiment_label(item["experiment_id"]), "Status": item["status"]}
        for item in experiments
    ]
    + [
        {"Experiment": record["label"], "Status": record["status"]}
        for record in detectors
        if record["family"] != "RT-DETRv2"
    ]
)

## 1. Timing probe

The probe measures the cost of a few real updates before the full runs and checks the
conventions on the GPU. Projections are linear extrapolations, not results.

In [ ]:
probe = records["probe"]
if probe is None:
    display(Markdown("**Not run.** The timing probe has no record."))
else:
    check = probe["ignore_check"]
    display(
        pl.DataFrame(
            {
                "Measurement": [
                    "Device",
                    "Transformers",
                    "Loss relative difference to Hugging Face",
                    "Training images per second",
                    "Seconds per update",
                    "Peak GPU MiB",
                    "Inference ms per frame, batch 1",
                    "Ignore frame: objects and matched queries",
                    "Ignore frame: dropped queries in the final layer",
                    "Object size in model pixels, median",
                    "NWD constant",
                    "Projected training minutes",
                    "Projected validation minutes",
                ],
                "Value": [
                    probe["gpu"] or probe["device"],
                    probe["transformers"],
                    f"{probe['reference_loss_check']['relative_difference']:.2e}",
                    f"{probe['train_images_per_second']:.2f}",
                    f"{probe['train_seconds_per_update']:.2f}",
                    f"{probe['peak_gpu_mib']:.0f}",
                    f"{probe['inference']['batch_1']['milliseconds_per_frame']:.1f}",
                    f"{check['objects']} and {check['matched_queries']:.0f}",
                    f"{check['excluded_queries_final']:.0f}",
                    f"{probe['object_size_model_pixels']['median']:.1f}",
                    f"{probe['nwd_constant']:.1f}",
                    f"{probe['projection']['train_minutes']:.0f}",
                    f"{probe['projection']['validation_minutes']:.0f}",
                ],
            }
        )
    )

## 2. Compute budget

The budget is the number of training image exposures on the same T4 GPU type. Training
minutes are the sum of end-to-end batch latencies and exclude setup, validation,
checkpoints and time between sessions, for every detector alike.

In [ ]:
budget_rows = [
    {
        "Detector": experiment_label(item["experiment_id"]),
        "Status": item["status"],
        "Image exposures": item.get("image_exposures"),
        "Optimizer updates": item.get("global_step"),
        "Training minutes": (
            round(item["wall_train_seconds"] / 60, 1) if "wall_train_seconds" in item else None
        ),
    }
    for item in experiments
] + [
    {
        "Detector": record["label"],
        "Status": record["status"],
        "Image exposures": record["image_exposures"],
        "Optimizer updates": None,
        "Training minutes": (
            round(record["train_minutes"], 1) if record["train_minutes"] is not None else None
        ),
    }
    for record in detectors
    if record["family"] != "RT-DETRv2"
]
budget = pl.from_dicts(budget_rows, infer_schema_length=None).with_columns(
    (pl.col("Image exposures") / (pl.col("Training minutes") * 60)).round(2).alias("Images/s")
)
budget

## 3. Validation of the final models

We compare the RT-DETRv2 experiments at their last completed epoch at maxDets 100 and 500.
The inference time at batch size 1 covers decoding, transfers, resizing, the model and the
top-k selection; there is no NMS step.

In [ ]:
final_rows = [
    {
        "Experiment": experiment_label(item["experiment_id"]),
        "Epoch": item["epochs"][-1]["epoch"],
        "mAP, maxDets 100": item["epochs"][-1]["map_100"],
        "AP50, maxDets 100": item["epochs"][-1]["ap50_100"],
        "AP small, maxDets 100": item["epochs"][-1]["ap_small_100"],
        "mAP, maxDets 500": item["epochs"][-1]["map_500"],
        "AP small, maxDets 500": item["epochs"][-1]["ap_small_500"],
        "Inference ms/frame, batch 1": (
            item["speed"]["milliseconds_per_frame"] if item.get("speed") else None
        ),
    }
    for item in experiments
    if item["epochs"]
]
if final_rows:
    display(pl.from_dicts(final_rows, infer_schema_length=None))
else:
    display(Markdown("**Not run.** No RT-DETRv2 validation has been recorded."))

## 4. Validation mAP over training against the CNN detectors

Each point is one full validation after an epoch. The first chart counts image exposures,
the budget definition; the second counts training minutes. Dashed lines mark the total
training minutes of Faster R-CNN B1 and of the standard-anchor RetinaNet run.

In [ ]:
curve_rows = [
    {"Detector": record["label"], **row} for record in detectors for row in record["curve"]
]
if not detectors:
    curve_rows = [
        {
            "Detector": experiment_label(item["experiment_id"]),
            "epoch": row["epoch"],
            "image_exposures": row["epoch"] * item["train_frames"],
            "train_minutes": None,
            "map": row["map_100"],
        }
        for item in experiments
        for row in item["epochs"]
    ]
not_run = [record["label"] for record in detectors if record["status"] == "Not run"]
if not_run:
    display(Markdown("**Not run:** " + ", ".join(not_run)))
if curve_rows:
    curves = pl.from_dicts(curve_rows, infer_schema_length=None)
    px.line(
        curves,
        x="image_exposures",
        y="map",
        color="Detector",
        markers=True,
        title="Validation mAP by image exposures",
        labels={"image_exposures": "Training image exposures", "map": "mAP, maxDets 100"},
    ).show(renderer="plotly_mimetype", config={"responsive": False})
    timed = curves.filter(pl.col("train_minutes").is_not_null())
    if timed.height:
        figure = px.line(
            timed,
            x="train_minutes",
            y="map",
            color="Detector",
            markers=True,
            title="Validation mAP by training minutes on a T4",
            labels={"train_minutes": "Training minutes", "map": "mAP, maxDets 100"},
        )
        for record in detectors:
            if (
                record["key"] in {"faster_rcnn_b1", "retinanet_standard_stride5"}
                and record["train_minutes"]
            ):
                figure.add_vline(
                    x=record["train_minutes"],
                    line_dash="dash",
                    annotation_text=record["label"],
                )
        figure.show(renderer="plotly_mimetype", config={"responsive": False})
else:
    display(Markdown("**Not run.** No validation curves are available."))

## 5. Training dynamics

The loss curves are batch means over windows of at most 100 attempts. Each part sums its
varifocal, L1 and box terms: the final decoder layer, the earlier decoder layers, the
encoder proposals, and the denoising queries. The diagnostics show the matched queries,
the queries dropped by the ignore rule, and the gradient norm before clipping.

In [ ]:
history_columns = (*LOSS_COLUMNS, *DIAGNOSTIC_COLUMNS, "gradient_norm", "learning_rate")
for item in experiments:
    history_path = batch_dir / item["experiment_id"] / "history.json"
    if not history_path.is_file():
        continue
    label = experiment_label(item["experiment_id"])
    curve = aggregate_loss_curve(json.loads(history_path.read_text()), columns=history_columns)
    parts = curve.select(
        "epoch",
        "step",
        *[
            pl.sum_horizontal([f"loss_{term}_{part}" for term in LOSS_TERMS]).alias(part)
            for part in LOSS_PARTS
        ],
    ).unpivot(index=["epoch", "step"], variable_name="Part", value_name="Mean loss")
    px.line(
        parts,
        x="step",
        y="Mean loss",
        color="Part",
        title=f"Training loss by part: {label}",
        labels={"step": "Optimizer update"},
    ).show(renderer="plotly_mimetype", config={"responsive": False})
    diagnostics = curve.select(
        "step", "matched_queries", "excluded_queries_final", "gradient_norm"
    ).unpivot(index="step", variable_name="Diagnostic", value_name="Mean value")
    px.line(
        diagnostics,
        x="step",
        y="Mean value",
        facet_row="Diagnostic",
        title=f"Matching and gradient diagnostics: {label}",
        labels={"step": "Optimizer update"},
        height=650,
    ).update_yaxes(matches=None).show(renderer="plotly_mimetype", config={"responsive": False})

## 6. Duplicates without NMS

A detector with class-wise NMS at IoU $t$ cannot output two same-class boxes above $t$.
RT-DETRv2 has no NMS, so the counts below show how often its queries predict the same
object twice. The second table re-evaluates each RT-DETRv2 run after adding class-wise NMS.

In [ ]:
duplicate_rows = [
    {
        "Detector": record["label"],
        "Score at least": row["score_threshold"],
        "IoU at least": row["iou_threshold"],
        "Detections": row["detections"],
        "Duplicate pairs": row["duplicate_pairs"],
        "Pairs per 100 detections": row["pairs_per_100_detections"],
        "Images with pairs": row["images_with_pairs"],
    }
    for record in detectors
    if "predictions" in record
    for row in record["predictions"]["near_duplicates"]
]
nms_rows = [
    {
        "Detector": record["label"],
        "mAP without NMS": record["predictions"]["area_bins"]["all"],
        **{
            f"mAP with NMS {row['iou_threshold']}": row["map"]
            for row in record["predictions"]["post_hoc_nms"]
        },
    }
    for record in detectors
    if "predictions" in record and "post_hoc_nms" in record["predictions"]
]
if duplicate_rows:
    display(pl.from_dicts(duplicate_rows, infer_schema_length=None))
    display(pl.from_dicts(nms_rows, infer_schema_length=None))
else:
    display(Markdown("**Not run.** The comparison analysis has no record."))

## 7. Small objects

mAP by ground-truth area in original-image pixels, at maxDets 100. The bins split the COCO
small range, below $32^2$, into three finer bins, because most VisDrone objects are small.

In [ ]:
area_rows = [
    {"Detector": record["label"], "Area bin": name, "mAP": value}
    for record in detectors
    if "predictions" in record
    for name, value in record["predictions"]["area_bins"].items()
]
if area_rows:
    areas = pl.from_dicts(area_rows, infer_schema_length=None)
    display(areas.pivot(on="Area bin", index="Detector", values="mAP"))
    px.bar(
        areas.filter(pl.col("Area bin") != "all"),
        x="Area bin",
        y="mAP",
        color="Detector",
        barmode="group",
        title="mAP by object area",
    ).show(renderer="plotly_mimetype", config={"responsive": False})
else:
    display(Markdown("**Not run.** The comparison analysis has no record."))

## 8. Dense scenes

Validation frames grouped by their number of target objects, crowd records excluded.

In [ ]:
density_rows = [
    {"Detector": record["label"], "Objects per frame": name, **values}
    for record in detectors
    if "predictions" in record
    for name, values in record["predictions"]["density_bins"].items()
]
if density_rows:
    density = pl.from_dicts(density_rows, infer_schema_length=None)
    display(density.pivot(on="Objects per frame", index="Detector", values="map"))
    display(density.select("Objects per frame", "images", "objects").unique(maintain_order=True))
    px.line(
        density,
        x="Objects per frame",
        y="map",
        color="Detector",
        markers=True,
        title="mAP by scene density",
        labels={"map": "mAP, maxDets 100"},
    ).show(renderer="plotly_mimetype", config={"responsive": False})
else:
    display(Markdown("**Not run.** The comparison analysis has no record."))

## 9. Convergence within the budget

For every detector: the best epoch, the mAP change over the last epoch, the relative change
of the mean training loss between the last two epochs, and whether the last epoch still
added more than half a point of mAP.

In [ ]:
convergence_rows = [{"Detector": record["label"], **record["convergence"]} for record in detectors]
if convergence_rows:
    display(pl.from_dicts(convergence_rows, infer_schema_length=None))
else:
    display(Markdown("**Not run.** The comparison analysis has no record."))

## 10. Class AP at the last completed epoch

In [ ]:
class_rows = [
    {"Class": name, "Experiment": experiment_label(item["experiment_id"]), "AP": value}
    for item in experiments
    if item["epochs"]
    for name, value in item["per_class_ap_100"].items()
]
if class_rows:
    display(pl.from_dicts(class_rows).pivot(on="Experiment", index="Class", values="AP"))
else:
    display(Markdown("**Not run.** No RT-DETRv2 validation has been recorded."))

## 11. Evaluation boundaries and reproduction

COCO AP with the original category-specific crowd records stands in for the competition
metric, whose exact ignore semantics are not given in the originals. pycocotools applies
maxDets per image and category. Every result comes from one seed, and CUDA kernels can make
reruns differ slightly.

To rerun this notebook, download `rt_detr_records.zip` from
`hm2_retinanet/batches/rt_detr_v2_r50_stride5_5epochs_seed2026/_analysis/` on Drive (or
from the output of the Kaggle analysis run) and extract it in the repository root:

```bash
uv run python -m zipfile -e /path/to/rt_detr_records.zip .
```

The records land in the gitignored `runs/rt_detr/<batch id>/`; set `RT_DETR_BATCH_DIR` to
read them from elsewhere.